# TCC - Mara Oliveira - Projeto 04

## Impacto dos fatores sociodemográficos na taxa de conclusão de curso nos Institutos Federais de Educação no Brasil

### Bibliotecas

In [ ]:
import sys
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# 1. Configurações gerais
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# 2. Ativação do autoreload via API do IPython (compatível e sem SyntaxError no VS Code)
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

# 3. Adiciona a pasta scripts ao caminho do Python ANTES de importar os módulos locais
sys.path.append(str(Path.cwd().parent / 'scripts'))

# 4. Importação dos módulos do seu projeto
from prepara_dados_fator import prepara_dados_fator
from executa_teste_quiquadrado import executa_teste_quiquadrado
from verifica_pressupostos_anova import verifica_pressupostos_anova
from executa_anova_oneway import executa_anova_oneway
from executa_kruskal_wallis import executa_kruskal_wallis
from executa_tukey_hsd import executa_tukey_hsd
from executa_posthoc_mannwhitney import executa_posthoc_mannwhitney
from executa_posthoc_quiquadrado import executa_posthoc_quiquadrado
from plota_boxplot_fator import plota_boxplot_fator
from plota_barras_horizontais import plota_barras_horizontais
from formata_tabela import formata_tabela
from gera_agrup_poshoc import gera_agrup_poshoc



### Arquivo de dados

In [ ]:
df = pd.read_parquet('../Data/Candidatos_Tratados.parquet')

### Etapas

As seguintes etapas serão seguidas para todos os fatores analisados:

#### Etapa 1. Análise Estatística Descritiva por Instituto X Geral

Nesta etapa, é realizado o agrupamento das taxas de conclusão de cada Instituto Federal (nível institucional). Os resultados são organizados em uma tabela comparando as medidas descritivas (mínimo, mediana, média simples e desvio-padrão dos IFs) com a Taxa Média Geral de Conclusão dos alunos na Rede Federal.

#### Etapa 2. Teste Qui-Quadrado 

O Teste do Qui-quadrado considera que cada estudante possui o mesmo peso na análise, independentemente se unidade IF de origem ter 500 ou 50 mil alunos. O objetivo é responder se a probabilidade de conclusão ou evasão do aluno está estatisticamente associada ao fator sociodemográfico avaliado.

#### Etapa 3. Gráfico de Barras Horizontais das Médias Ponderadas

Elaboração de gráfico de barras horizontais para comparação das médias de cada classe do fator, independemente da unidade do IF. Desta forma, cada barra no gráfico representa a média ponderada da taxa de conclusão por fator. Sendo assim, IFs com maior número de alunos influenciam mais no cálculo da média.

#### Etapa 4. Teste de normalidade

O teste de Normalidade avalia se os pressupostos de Gauss-Markov (normalidade dos resíduos via Shapiro-Wilk e homogeneidade das variâncias via Levene) são atendidos pelo fator. 

#### Etapa 5. ANOVA One-Way ou Teste de Kruskal-Wallis

Conforme os pressupostos de Gauss-Markov são ou não atendidas, opta-se por usar para a análise  o método paramétrico (Anova de Fisher) ou o não-paramétrico (Teste de Kruskal-Wallis ou ANOVA por postos). Ambos os testes avaliam se a influência do fator sobre as taxas institucionais é estatisticamente significativa. O que diferencia estes dois métodos estatísticos do Qui-Quadrado é a base de observação. Enquanto no Qui-Quadrado, a base é o estudante, na ANOVA e no Kruskal-Wallis, a base é a taxa de cada Instituto Federal, tendo cada instituição o mesmo peso independente do número de estudantes matriculados.

#### Etapa 6. Teste de Comparação de Médias

A medida que os testes anteriores (Anova de Fisher ou Teste de Kruskal_Wallis) indicarem diferença estatisticamente signifcativa ($p < 0,05$), a próxima análise e o Teste de Comparação de Médias. Este teste tem a finalidade de identificar entre quais categorias do fator se as diferenças foram estatísticamene significativas. Para isso, utiliza-se o Teste de Tukey HSD, se o modelo for a ANOVA - paramétrica, ou o Teste de Mann-Whitney com correção de Bonferroni, se o modelo for o Kruskal-Wallis - não-paramétrico.

#### Etapa 7. Gráfico de Barras Horizontais das Médias por IF

Elaboração de gráfico de barras horizontais para mostrar as médias de cada classe do fator, considerando como unidade de observação a média da taxa de conclusão de cada unidade de IF por fator. Neste caso, cada barra representa a média das médias da taxa de aprovação por unidade do IF. Sendo assim, a 

#### Etapa 8. Gráfico de Boxplot com Strip Plot das Médias por IF

Elaboração de gráfico boxplot para visualização da dispersão das taxas das instituições, destacando as medianas, a média de cada grupo, cada IF individualizado e a linha de referência da média nacional.


    

In [ ]:
fatores = ['REGIAO', 'ETNIA', 'FAIXA_ETARIA', 'RENDA', 'SEXO']

for fator in fatores:

    # Cabeçalho
    print("\n")
    print("=" * 100)
    if len(fator) % 2 == 0:
        print(str("-" * (49 - int(len(fator)/2))) + " " + f"{fator}" + " " + str("-" * (49 - int(len(fator)/2))))
    else:
        print(str("-" * (49 - int(len(fator)/2))) + " " + f"{fator}" + " " + str("-" * (48 - int(len(fator)/2))))
    print("=" * 100)
    print("\n")

    # -------------------------------------------------------------
    # BLOCO 1: NÍVEL DISCENTE (ESTUDANTES)
    # -------------------------------------------------------------
    print("\nETAPA 01 - ANÁLISE ESTATÍSTICA DESCRITIVA\n")
    df_inst, tab_fator = prepara_dados_fator(df, fator)

    print("Tabela 01. Análise Estatística Descritiva")
    formata_tabela(tab_fator, fator)

    print("\nETAPA 02. TESTE QUI-QUADRADO E V DE CRAMÉR\n")
    tab_chi, chi2, p_chi2, v_cramer, intensidade = executa_teste_quiquadrado(df, fator)

    print(f"Qui-Quadrado Global: χ² = {chi2:,.2f} | p-valor = {p_chi2:.4e}")
    print(f"Tamanho do Efeito Global (V de Cramér): V = {v_cramer:.4f} -> Associação {intensidade}")

    # Pós-teste par a par do Qui-Quadrado (apenas se k > 2 e significante)
    df_post_chi = None
    if p_chi2 < 0.05 and df[fator].nunique() > 2:
        df_post_chi = executa_posthoc_quiquadrado(df, fator)

    # Agrupamento Pós-hoc do Qui-Quadrado (Nível Discente)
    ordem_chi = tab_chi[fator].tolist()
    cld_chi, paleta_chi = gera_agrup_poshoc(
        fator=fator,
        df_posthoc=df_post_chi,
        p_global=p_chi2,
        ordem=ordem_chi,
        filtrar_residual=True
    )

    tab_chi['GRUPO'] = tab_chi[fator].map(cld_chi)
    print(f"\nTabela 02. Análise de Contingência para {fator} (Nível Estudante)")
    formata_tabela(tab_chi, fator)

    print(f"\nETAPA 03. GRÁFICO DA TAXA MÉDIA GERAL DE CONCLUSÃO POR {fator}\n")
    plota_barras_horizontais(df, tab_chi, fator, fator, stat='TAXA_MEDIA_GERAL', paleta_cores=paleta_chi)

    # -------------------------------------------------------------
    # BLOCO 2: NÍVEL INSTITUCIONAL (INSTITUTOS FEDERAIS)
    # -------------------------------------------------------------
    print("\nETAPA 04. VERIFICAÇÃO DOS PRESSUPOSTOS PARA ANOVA (GAUSS-MARKOV)\n")
    p_shapiro, p_levene = verifica_pressupostos_anova(df_inst, fator)
    print(f"p-valor Shapiro-Wilk (Normalidade): {p_shapiro:.4e}")
    print(f"p-valor Levene (Homocedasticidade): {p_levene:.4e}")

    # Etapas 05 e 06: Teste Global + Pós-teste
    df_post = None
    if p_shapiro > 0.05 and p_levene > 0.05:
        print(f"\nETAPA 05. ANOVA ONE-WAY (PARAMÉTRICA) PARA {fator}\n")
        f_stat, p_global = executa_anova_oneway(df_inst, fator)
        print(f"ANOVA: F = {f_stat:.4f} | p-valor = {p_global:.4e}")

        if p_global < 0.05:
            print(f"\nETAPA 06. TESTE (TUKEY HSD) DE COMPARAÇÕES DAS TAXAS MÉDIAS DE CONCLUSÃO DOS IFs PARA {fator}\n")
            df_post = executa_tukey_hsd(df_inst, fator)
        else:
            print(f"Não há diferença estatisticamente significante entre as médias institucionais de {fator} (p >= 0.05).")
    else:
        print(f"\nETAPA 05. TESTE DE KRUSKAL-WALLIS (NÃO-PARAMÉTRICO) PARA {fator}\n")
        h_stat, p_global = executa_kruskal_wallis(df_inst, fator)
        print(f"Kruskal-Wallis: H = {h_stat:.4f} | p-valor = {p_global:.4e}\n")

        if p_global < 0.05:
            if df_inst[fator].nunique() > 2:
                print(f"\nETAPA 06. TESTE (MANN-WHITNEY COM BONFERRONI) DE COMPARAÇÕES DAS TAXAS MÉDIAS POR IFs PARA {fator}\n")
                df_post = executa_posthoc_mannwhitney(df_inst, fator)
            else:
                print(f"Como {fator} possui apenas 2 categorias, o teste global já compara diretamente ambos os grupos.")
        else:
            print(f"Não há diferença estatisticamente significante entre as medianas institucionais de {fator} (p >= 0.05).")

    # Agrupamento Pós-hoc Institucional (funciona universalmente para ANOVA, Kruskal, k=2 ou p>=0.05)
    cld_map, paleta_cores = gera_agrup_poshoc(
        df_inst=df_inst,
        fator=fator,
        df_posthoc=df_post,
        p_global=p_global
    )

    # Tabela descritiva final com a coluna GRUPO (Compact Letter Display)
    print(f"\nTabela 04. Estatística Descritiva Institucional para {fator} (Nível Instituição)")
    tab_fator['GRUPO'] = tab_fator[fator].map(cld_map)
    formata_tabela(tab_fator, fator)

    print(f"\nETAPA 07. GRÁFICO DA TAXA MÉDIA DE CONCLUSÃO DOS IFs POR {fator}\n")
    plota_barras_horizontais(df, tab_fator, fator, fator, stat='TAXA_MEDIA_IF', df_inst=df_inst, paleta_cores=paleta_cores)

    print(f"\nETAPA 08. GRÁFICO DA DISTRIBUIÇÃO DAS TAXAS MÉDIAS DE CONCLUSÃO DOS IFs POR {fator}\n")
    plota_boxplot_fator(df, df_inst, fator, fator, paleta_cores=paleta_cores)
